In [ ]:
!date

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc'

donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
tmp_order = ['Start', 'Sendai', 'Fail1', 'Delayed', 'Inter1', 'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/chrom100k/merged_less_obs.h5ad')

In [ ]:
adata

In [ ]:
sc.pl.umap(adata, color=['line', 'time'])

In [ ]:
adata.obs

In [ ]:
projdir

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna'
clusters = pd.read_csv(f'{indir}/csv/label_transfer/snmct_xgboost_v7.csv', sep='\t', index_col=0)
clusters.shape

In [ ]:
clusters.head()

In [ ]:
adata.obs['cluster'] = clusters.reindex(adata.obs.index)['pred_coarse']

In [ ]:
sc.pl.umap(adata, color=['line', 'cluster'])

In [ ]:
tmp_adata = adata[~adata.obs['cluster'].isna()]
tmp_adata.shape

In [ ]:
tmp_adata.obs.head()

In [ ]:
final_clusters = clusters.reindex(tmp_adata.obs.index)
final_clusters.shape

In [ ]:
final_clusters.head()

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna'
final_clusters.to_csv(f'{outdir}/csv/label_transfer/snmct_xgboost_v7_filtered.csv', sep='\t')

### donor by donor

In [ ]:
leiden_path = f'{projdir}/csv/integrate_m3c/leiden_clusters_renamed.csv'
leiden = pd.read_csv(leiden_path, sep='\t', index_col=0)
leiden.head()
leiden.shape

In [ ]:
adatas = []
for d in donors:
    donor_adata = sc.read_h5ad(f'{projdir}/h5ad/chrom100k/{d}.h5ad')
    donor_adata.obs['cluster'] = clusters.reindex(donor_adata.obs.index)['pred_coarse']
    donor_adata.obs['leiden'] = leiden.reindex(donor_adata.obs.index)['renamed_leiden']
    adatas.append(donor_adata)

In [ ]:
for donor_adata in adatas:
    sc.pl.umap(donor_adata, color=['cluster', 'leiden'])

### what about without PMDs?

In [ ]:
nonpmd_adatas = []
for d in donors:
    donor_adata = sc.read_h5ad(f'{projdir}/h5ad/chrom100k/{d}.remove_pmds.h5ad')
    donor_adata.obs.index = [x.replace('_indexed', '') for x in donor_adata.obs.index]
    donor_adata.obs['cluster'] = clusters.reindex(donor_adata.obs.index)['pred_coarse']
    donor_adata.obs['leiden'] = leiden.reindex(donor_adata.obs.index)['renamed_leiden']
    nonpmd_adatas.append(donor_adata)

In [ ]:
for donor_adata in nonpmd_adatas:
    sc.pl.umap(donor_adata, color=['cluster', 'leiden'])

In [ ]:
!date